# C04. 펭귄 종 분류 — 4차시 코드에서 두 줄만 바꾸자

> 📌 **이 모듈에서 할 것**
> 
> 4차시 다변량 회귀 코드를 그대로 가져와서, **두 줄만 바꿔서 분류 모델**로 만들기.

## 사전 지식

- C01, C02, C03 모두 완료
- 4차시 (PyTorch 다변량 회귀) 코드 익숙

---


## 1. 데이터 준비 — 4차시와 거의 동일


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

# 데이터 생성 (실제 환경에선 seaborn 사용)
np.random.seed(42)
species_stats = {
    'Adelie':    {'n': 152, 'bill_len': (38.8, 2.7), 'bill_dep': (18.3, 1.2),
                  'flipper': (190, 6.5)},
    'Gentoo':    {'n': 124, 'bill_len': (47.5, 3.1), 'bill_dep': (14.98, 0.98),
                  'flipper': (217, 6.5)},
}
rows = []
for sp, s in species_stats.items():
    for _ in range(s['n']):
        size = np.random.normal(0, 1)
        rows.append({
            'species': sp,
            'bill_length_mm':  s['bill_len'][0] + s['bill_len'][1] * (size * 0.4 + np.random.normal(0, 0.6)),
            'bill_depth_mm':   s['bill_dep'][0] + s['bill_dep'][1] * (size * 0.4 + np.random.normal(0, 0.6)),
            'flipper_length_mm': s['flipper'][0] + s['flipper'][1] * (size * 0.7 + np.random.normal(0, 0.4)),
        })
penguins = pd.DataFrame(rows)
print(f"총 {len(penguins)}마리 (Adelie + Gentoo)")
penguins.head()

# Colab에서는 실제 데이터:
# import seaborn as sns
# penguins = sns.load_dataset('penguins').dropna()
# penguins = penguins[penguins['species'].isin(['Adelie', 'Gentoo'])]


In [ ]:
# 정규화 + 텐서
def normalize(s):
    return (s - s.mean()) / s.std()

features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
X = torch.tensor(
    np.column_stack([normalize(penguins[c]).values for c in features]),
    dtype=torch.float32
)

# 라벨: Adelie=0, Gentoo=1
y = torch.tensor((penguins['species'] == 'Gentoo').values, dtype=torch.float32)

print(f"X: {X.shape}, y: {y.shape}")
print(f"y 분포: Adelie={int((y==0).sum())}, Gentoo={int((y==1).sum())}")


## 2. 분류 학습 코드 — 4차시 회귀에서 **두 줄만** 바꿈

```python
# 4차시 회귀:
y_hat = X @ w + b
loss = ((y - y_hat) ** 2).mean()

# C04 분류:
y_hat = torch.sigmoid(X @ w + b)    # ← sigmoid 추가
loss = F.binary_cross_entropy(y_hat, y)   # ← cross-entropy로 변경
```

나머지는 완전히 동일!


In [ ]:
# 학습 — 4차시 코드 구조 그대로
w = torch.zeros(3, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
eta = 0.1
epochs = 1000

losses = []
for epoch in range(epochs):
    # 두 줄만 다름 ↓
    y_hat = torch.sigmoid(X @ w + b)
    loss = F.binary_cross_entropy(y_hat, y)
    
    # 나머지는 동일
    loss.backward()
    with torch.no_grad():
        w -= eta * w.grad
        b -= eta * b.grad
        w.grad.zero_()
        b.grad.zero_()
    losses.append(loss.item())
    
    if epoch % 100 == 0:
        print(f"epoch {epoch:4d}: loss={loss.item():.4f}")

print(f"\n최종 가중치 w = {w.detach().numpy()}")
print(f"편향 b = {b.item():.4f}")


## 3. 학습 곡선


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Training Loss (Penguin Binary Classification)')
plt.grid(alpha=0.3)
plt.show()


## 4. 정확도 측정


In [ ]:
# 예측 확률 → 0/1 라벨
with torch.no_grad():
    probs = torch.sigmoid(X @ w + b)
    predictions = (probs > 0.5).float()

# 정확도
accuracy = (predictions == y).float().mean().item()
print(f"훈련 데이터 정확도: {accuracy*100:.1f}%")


## 5. 결정경계 시각화

분류 모델은 입력 공간을 **두 영역으로 나누는 결정경계**를 학습해요.  
2차원으로 보기 위해 두 피처만 사용해서 다시 학습:


In [ ]:
# 2차원으로 단순화 (bill_length, flipper_length만)
X_2d = torch.tensor(
    np.column_stack([
        normalize(penguins['bill_length_mm']).values,
        normalize(penguins['flipper_length_mm']).values,
    ]),
    dtype=torch.float32
)

w2 = torch.zeros(2, requires_grad=True)
b2 = torch.tensor(0.0, requires_grad=True)
eta = 0.1
epochs = 1000

for epoch in range(epochs):
    y_hat = torch.sigmoid(X_2d @ w2 + b2)
    loss = F.binary_cross_entropy(y_hat, y)
    loss.backward()
    with torch.no_grad():
        w2 -= eta * w2.grad
        b2 -= eta * b2.grad
        w2.grad.zero_()
        b2.grad.zero_()

print(f"2D 학습 완료: w = {w2.detach().numpy()}, b = {b2.item():.4f}")


In [ ]:
# 결정경계 + 데이터 시각화
xx, yy = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))
grid = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)
with torch.no_grad():
    probs_grid = torch.sigmoid(grid @ w2 + b2).numpy().reshape(xx.shape)

plt.figure(figsize=(8, 6))
# 결정경계 영역 (확률 색칠)
plt.contourf(xx, yy, probs_grid, levels=20, cmap='RdYlBu_r', alpha=0.5)
plt.colorbar(label='P(Gentoo)')
plt.contour(xx, yy, probs_grid, levels=[0.5], colors='black', linewidths=2)

# 데이터 점
colors = {'Adelie': '#FF8C00', 'Gentoo': '#008B8B'}
for sp, c in colors.items():
    mask = (penguins['species'] == sp).values
    plt.scatter(X_2d[mask, 0].numpy(), X_2d[mask, 1].numpy(),
                c=c, label=sp, s=30, edgecolors='black', linewidths=0.5)

plt.xlabel('Bill Length (normalized)')
plt.ylabel('Flipper Length (normalized)')
plt.title('Decision Boundary (black line = 50% probability)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**검은 선이 결정경계** (모델이 "여기 기준으로 두 종을 나눈다"고 학습한 경계).  
파란 영역은 "Gentoo일 가능성 높음", 빨강은 "Adelie".

학습이 잘 됐다면 두 색깔의 점이 검은 선을 기준으로 깔끔하게 갈라져 있어요.


## 6. 회귀와 분류 — 한눈에 비교

같은 펭귄 데이터, 두 다른 질문, 거의 같은 코드:

| | 회귀 (4차시) | 분류 (C04) |
|---|---|---|
| 타겟 | `body_mass_g` (수치) | `species` (Adelie/Gentoo) |
| y 변환 | 정규화 | (Adelie=0, Gentoo=1) |
| 예측 | `y_hat = X @ w + b` | `y_hat = sigmoid(X @ w + b)` |
| 손실 | `((y - y_hat)**2).mean()` | `F.binary_cross_entropy(y_hat, y)` |
| 결과 시각화 | 예측 vs 실제 산점도 | 결정경계 + 색칠 |


## 7. 본인 데이터에 적용해보기 ✏️

다른 이진 분류 문제를 해보세요. 예시:

- 학생 데이터: 점수로 합격/불합격 분류
- 펭귄 다중 분류: Adelie vs Chinstrap vs Gentoo (소프트맥스 필요)
- 본인이 모은 데이터: 어떤 카테고리 예측

```python
# 1. 데이터 준비
# y = (조건).astype(int)  # 0 또는 1

# 2. 정규화 + 텐서
# X = normalize and to tensor

# 3. 학습 (위 코드 그대로 복사)
# y_hat = torch.sigmoid(X @ w + b)
# loss = F.binary_cross_entropy(y_hat, y)
```


## 8. ⚠️ 함정 / 주의사항

### 8.1 라벨이 float인지 확인
`binary_cross_entropy`는 float 라벨을 받음. int이면 에러.

### 8.2 클래스 불균형
한쪽이 90%, 다른 쪽이 10%면 모델이 다수만 맞춰도 정확도 90%.  
**해결**: precision, recall, F1 score 같이 보기.

### 8.3 결정경계는 선형
`sigmoid(X @ w + b)`는 **선형 결정경계**만 그림.  
복잡한 경계(곡선) 필요하면 신경망 (다음 학기 주제).


## 9. 📚 더 알아보기

- **다중 분류**: 3개 이상 클래스 (`F.cross_entropy`, softmax)
- **신경망**: 비선형 결정경계 (히든 레이어)
- **평가 지표**: confusion matrix, ROC curve, AUC
- **정규화 (regularization)**: L1/L2로 과적합 방지

---

## 🎉 분류 보충 트랙 완주!

회귀와 분류가 **거의 같은 구조**라는 걸 직접 확인했어요.  
PyTorch에선 코드 두 줄 차이.

진짜 깊이 가려면 다음 학기에 1·2차시처럼 수학적으로 유도해봅시다.
